# Migrate numbered Neon RAW exports

Moves existing `Neon/<recording_num>` contents into `Neon/<recording_num>/timeseries_and_video_export`. The `native_recording_data` folder stays in place. Internal export folders and file contents are preserved. No downloads are performed.

Set `raw_dir`; target subjects and activities are loaded from the preprocessing pipeline’s subject and activity workbooks. Run the preview, then set `apply_moves = True` to move the listed entries. Run this before using the updated preprocessing pipeline on existing data. Stop any downloads or processing that use these folders first.


In [ ]:
import pathlib
import shutil
import preprocessing_pipeline

raw_dir = pathlib.Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")

# get_subject_ids returns {numeric subject ID: age}; its keys select subjects.
subjects_to_migrate = preprocessing_pipeline.get_subject_ids()
activities_to_migrate = preprocessing_pipeline.get_activity_names()
apply_moves = False


In [ ]:
def _migrate_neon_export(recording_dir: str) -> None:
    """Move a previous numbered export into the new format-specific folder.

    No files are deleted. If migration was interrupted, a subsequent call moves
    the remaining entries, refusing to replace any conflicting destination.
    """
    root = pathlib.Path(recording_dir)
    if not root.is_dir():
        return
    export = root / "timeseries_and_video_export"
    entries = [p for p in root.iterdir() if p.name not in (
        "timeseries_and_video_export", "native_recording_data")
        and not p.name.startswith(".pupil-download-")]
    if not entries:
        return
    if not export.is_dir() and not any(
        (p.is_file() and p.name == "world_timestamps.csv") or
        (p.is_dir() and any(p.rglob("world_timestamps.csv"))) for p in entries
    ):
        raise ValueError(f"Cannot identify legacy Neon export in {root}")
    for entry in entries:
        if (export / entry.name).exists():
            raise FileExistsError(f"Migration conflict: {export / entry.name}")
    export.mkdir(exist_ok=True)
    for entry in entries:
        shutil.move(str(entry), str(export / entry.name))




In [ ]:
# Preview and preflight all selected attempts before moving any data.
if not raw_dir.is_dir():
    raise FileNotFoundError(raw_dir)

migration_plan = []
for subject_id in sorted(subjects_to_migrate):
    subject = raw_dir / f"FLIC_{subject_id}"
    if not subject.is_dir():
        continue
    for activity_name in sorted(activities_to_migrate):
        activity = subject / activity_name
        if not activity.is_dir():
            continue
        neon = activity / "Neon"
        if not neon.is_dir():
            continue
        for attempt in sorted(neon.iterdir()):
            if not attempt.is_dir() or not attempt.name.isdigit():
                continue
            export = attempt / "timeseries_and_video_export"
            entries = [p for p in attempt.iterdir() if p.name not in (
                "timeseries_and_video_export", "native_recording_data")
                and not p.name.startswith(".pupil-download-")]
            if not entries:
                continue
            if export.exists() and not export.is_dir():
                raise NotADirectoryError(export)
            # Identify old CSV exports without looking inside native data.
            if not export.is_dir() and not any(
                (p.is_file() and p.name == "world_timestamps.csv") or
                (p.is_dir() and any(p.rglob("world_timestamps.csv")))
                for p in entries
            ):
                raise ValueError(f"Cannot identify legacy Neon export in {attempt}")
            for entry in entries:
                target = export / entry.name
                if target.exists():
                    raise FileExistsError(f"Migration conflict: {target}")
                print(f"{entry} -> {target}")
            migration_plan.append(attempt)

print(f"\n{len(migration_plan)} recording attempts to migrate. No files moved yet.")


In [ ]:
if not apply_moves:
    print("Dry run only. Set apply_moves = True and rerun this cell to migrate.")
else:
    for attempt in migration_plan:
        _migrate_neon_export(str(attempt))
        print(f"Migrated: {attempt}")
    print("Migration complete. Rerun the preview to verify no moves remain.")
